# Gradient descent and Newton's method

(This exercise originates from S. Boyd's course Convex Optimization)

Consider the unconstrained problem
  
  $$\begin{array}{ll}minimize\,\,\, f(x) = - \sum_{i=1}^m \log(1-a_i^T x)- \sum_{i=1}^n \log(1 - x_i^2),\end{array}$$
  
  with variable $x \in \mathbf{R}^n$, and $\mathbf{dom} f = \{x \;|\; a_i^Tx \le 1, ~i=1, \ldots, m, ~|x_i| \le 1, ~i=1, \ldots, n\}$. This is the problem of computing the analytic center of the set of linear inequalities
  
  $$a_i^Tx \leq 1, \quad i=1, \ldots, m, \qquad |x_i|\leq 1,\quad i=1, \ldots, n.$$
 
 Note that we can choose $x^{(0)}=0$ as our initial point. You can generate instances of this problem by choosing $a_i$ from some distribution on $\mathbf{R}^n$.
 

1. Use the gradient descent method to solve the problem. Use backtracking line search (BLS) to choose the step size and a stopping criterion of the form $\| \nabla f(x)\|_2^2 \leq \delta$. You need to choose the parameters $\alpha,\beta$ of BLS as well as $\delta$ in a reasonable way. Plot the objective function and step length versus iteration number. (Once you have determined $p^\star$ to high accuracy, you can also plot $f-p^\star$ versus iteration.) Experiment with the backtracking parameters $\alpha$ and $\beta$ to see their effect on the total number of iterations required. 
2. Repeat using Newton&#39;s method, again with BLS and this time with a stopping criterion based on the square $\lambda^2$ of Newton decrement. Produce the plots as for GD. Can you see the sharp distinction between the areas of slow and fast convergence for the Newton&#39;s method?

Some notes and tips:
* You can find the description of BLS in the slides for the Newton's method lecture. The exact same formulation can be used for GD.
* The domain of the objective is not $\mathbf{R}^n$. So, your BLS needs make sure that the chosen step length results in a point inside the domain (on top of satisfying the standard BLS condition). 
* You are going to have to find formulas for $\nabla f(x)$ and $\nabla^2 f(x)$ by hand using chain rule. If you really want to, you can use pytorch (or other similar tools) for this task, but it is probably not worth it. The GD code and the Newton code has to be your own.
* In the Newton's algorithm do not invert the Hessian, use ``numpy.linalg.solve`` (or other linear system solver) instead.

# Solution

## Objective Derivatives

$$ f(x) = - \sum_{k=1}^m \log(1-a_k^T x)- \sum_{i=1}^n \log(1 - x_i^2) $$

### Gradient

The formula for partial derivatives is:

$$ \frac{\partial f}{\partial x_i}(x) = \frac{2x_i}{1 - x_i^2} + \sum_{k=1}^m \frac{a_{ki}}{1-a_k^T x} $$

Hence, the gradient can be written as:

$$ \nabla f (x) = A^T \cdot d + b$$

where $A$ is an $m \times n$ matrix with $a_k^T$ as rows, $d$ is an $m$-dimensional column vector with $d_k = \frac{1}{1 - a_k^T x} > 0$ and $b$ is an $n$-dimensional column vector with $b_i = \frac{2x_i}{1 - x_i^2}$.


### Hessian

For $d(x)$ we have: 

$$ \frac{\partial d_i}{\partial x_j}(x) = \frac{a_{ij}}{(1 - a_i^Tx)^2} $$

which can be written in the matrix form as a Jacobian:

$$ J_d(x) = diag(\{d_i^2\}) A $$

Moreover for $b$, we have:

$$ \frac{\partial b_i}{\partial x_j}(x) = diag({\frac{2 + 2x_i^2}{(1 - x_i^2)^2}}) $$

Therefore:

$$ H_f = A^T diag(\{d_i^2\}) A + diag({\frac{2 + 2x_i^2}{(1 - x_i^2)^2}})$$

## Convexity of the Objective

Hessian can be rewritten as:

$$ H_f = \sum_i^m d_i^2 a_i a_i^T + diag({\frac{2 + 2x_i^2}{(1 - x_i^2)^2}}) $$

The sum part is a sum of positive semidefinite matrices, since $\forall v \in \mathbf{R}^n$:

$$ d_i^2 v^T (a_i a_i^T) v = d_i^2 ||v^T a_i||_2^2  \geq 0 $$

and therefore positive semidefinite.
The second term is a diagonal matrix with strictly positive terms, so it's stricly positive definite.
Hence $H_f \succ 0$ as a sum of a positive semidefinite (the sum part) and a strictly positive definite (the diagonal term) matrices.

# Programming Part

## Necessary imports

In [1]:
import numpy as np
import cvxpy as cp

## Custom implementation

In [2]:
m = 7500
n = 7400

np.random.seed(0)

A = np.random.rand(m, n)
x0 = np.zeros(n)

In [3]:
class ObjectiveFunction:
    def __init__(self, A):
        self.A = A
    
    def __call__(self, x):
        return -np.sum(np.log(1 - self.A @ x)) - np.sum(np.log(1 - np.square(x)))

    def gradient(self, x):
        return self.A.T @ (1 / (1 - self.A @ x)) + 2 * x / (1 - np.square(x))
    
    def hessian(self, x):
        d = 1 / (np.ones(self.A.shape[0]) - self.A @ x)
        # self.A.T @ D @ self.A with D = np.diag(d**2) but in a more efficient way
        first_term = self.A.T @ (d[:, None]**2 * self.A)
        second_term = np.diag(2 * (1 + np.square(x)) / (np.ones(self.A.shape[1]) - np.square(x))**2)
        return first_term + second_term

In [4]:
objective_function = ObjectiveFunction(A)

### General BLS

In [5]:
def bls(obj_func, x, search_direction, hard_max_step_size=1.0, softening_of_max_step_size = 0.99, alpha=0.5, beta=0.8):
    assert obj_func.gradient(x).T @ search_direction < 0, "incorrect search direction!"

    # A-related constraints
    #
    # We want:
    # A @ (x + max_step_size * search_direction) = A @ x + max_step_size * A @ search_driection <= 1
    # 
    # Hence we care only about these constraints for which a_i^T @ search_direction >= 0,
    # because only those can lead to violation of the constraints.

    nominators = 1 - obj_func.A @ x
    denominators = obj_func.A @ search_direction
    max_step_size = np.inf
    for i in range(len(denominators)):
        if denominators[i] <= 0:
            continue
        else:
            new_max_step_size_candidate = nominators[i] / denominators[i]
            if new_max_step_size_candidate < max_step_size:
                max_step_size = new_max_step_size_candidate

    # Box constraints
    #
    # |x_i + max_step_size * search_direction[i]| <= 1
    #
    # 1. If search_direction[i] > 0, we we solve:
    #
    # x_i + max_step_size * search_direction[i] <= 1
    #
    # which gives us:
    #
    # max_step_size <= (1 - x_i) / search_direction[i]
    #
    # 2. If search_direction[i] < 0, we can only decrease so we get a negative value on LHS, so:
    #
    # x_i + max_step_size * search_direction[i]| >= -1
    #
    # which gives us:
    #
    # max_step_size <= (-1 - x_i) / search_direction[i]

    for i in range(len(x)):
        if search_direction[i] > 0:
            new_max_step_size_candidate = (1 - x[i]) / search_direction[i]
        elif search_direction[i] < 0:
            new_max_step_size_candidate = (-1 - x[i]) / search_direction[i]
        else:
            new_max_step_size_candidate = np.inf
        
        if new_max_step_size_candidate < max_step_size:
            max_step_size = new_max_step_size_candidate

    step_size = np.min([hard_max_step_size, softening_of_max_step_size*max_step_size])
    
    while obj_func(x + step_size * search_direction) > obj_func(x) + alpha * step_size * np.dot(obj_func.gradient(x), search_direction):
        step_size = beta * step_size
    return step_size

### Gradient Descent BLS

In [6]:
def gd_bls(obj_func, x0, max_iters=1000, tol=1e-12, alpha=0.5, beta=0.8, log_int = 100):
    x = x0.copy()

    stopping_criterions = []
    function_values = []
    step_sizes = []

    for i in range(max_iters):
        if i % log_int == 0:
            # print(f"iter: {i}, current x: {x}")
            ...
        gradient = obj_func.gradient(x)
        search_direction = -gradient
        function_values.append(obj_func(x))
        stopping_criterion = gradient.T @ gradient
        stopping_criterions.append(stopping_criterion)
        step_size = bls(obj_func, x, search_direction, alpha=alpha, beta=beta)
        step_sizes.append(step_size)
        x = x + step_size * search_direction
        if stopping_criterion < tol:
            break
    
    return x, stopping_criterions, function_values, step_sizes


In [7]:
x0 = np.zeros(n)

In [8]:

x_sol_gd, s_crs_gd, fun_vs_gd, step_sizes_gd = gd_bls(objective_function, x0)

In [9]:
objective_function(x_sol_gd)

np.float64(-54503.11356553277)

### Newton's BLS

In [10]:
def newton_bls(obj_func, x0, max_iters=1000, tol=1e-12, alpha=0.5, beta=0.8, log_int = 100):
    x = x0.copy()

    stopping_criterions = []
    function_values = []
    step_sizes = []

    for i in range(max_iters):
        if i % log_int == 0:
            # print(f"iter: {i}, current x: {x}")
            ...
        gradient = obj_func.gradient(x)
        search_direction = -np.linalg.solve(obj_func.hessian(x), gradient)
        function_values.append(obj_func(x))
        lambda_sq_criterion = -gradient @ search_direction.T
        stopping_criterions.append(lambda_sq_criterion)
        step_size = bls(obj_func, x, search_direction, alpha=alpha, beta=beta)
        step_sizes.append(step_size)
        x = x + step_size * search_direction
        if lambda_sq_criterion < tol:
            break
    
    return x, stopping_criterions, function_values, step_sizes

In [11]:
x_sol_newton, s_crs_newton, fun_vs_newton, step_sizes_newton = newton_bls(objective_function, x0)

In [12]:
objective_function(x_sol_newton)

np.float64(-54503.11356553276)

### Plots

In [13]:
import plotly.graph_objects as go

In [14]:
steps_gd = list(range(len(s_crs_gd)))
steps_newton = list(range(len(s_crs_newton)))

In [15]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=steps_gd, y=fun_vs_gd, mode='lines+markers', name='GD'))
fig.add_trace(go.Scatter(x=steps_newton, y=fun_vs_newton, mode='lines+markers', name="Newton"))
fig.update_layout(
    title='Objective value vs. iteration',
    xaxis_title='iteration k',
    yaxis_title='f(x_k)',
)
fig.show()

In [16]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=steps_gd, y=step_sizes_gd, mode='lines+markers', name='GD'))
fig.add_trace(go.Scatter(x=steps_newton, y=step_sizes_newton, mode='lines+markers', name='Newton'))
fig.update_layout(
    title='Step length t vs. iteration',
    xaxis_title='iteration k',
    yaxis_title='step length t',
)
fig.show()

In [17]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=steps_gd, y=s_crs_gd, mode='lines+markers', name='GD: ||grad f||^2'))
fig.add_trace(go.Scatter(x=steps_newton, y=s_crs_newton, mode='lines+markers', name='Newton: lambda^2'))
fig.update_layout(
    title='Stopping criterion vs. iteration',
    xaxis_title='iteration k',
    yaxis_title='criterion (log scale)',
)
fig.update_yaxes(type='log')
fig.show()

The two plots above clearly show that the Newton's method enters fast convergence around 10th iteration for this problem instance. This obviously depends on the problem instance.

In [18]:
# Suboptimality f(x_k) - p* on a log scale.
pstar = objective_function(x_sol_newton)

# clip at a tiny floor so the log axis can render the last (near-zero / slightly negative) points
gap_gd = np.maximum(np.array(fun_vs_gd) - pstar, 1e-16)
gap_newton = np.maximum(np.array(fun_vs_newton) - pstar, 1e-16)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps_gd, y=gap_gd, mode='lines+markers', name='GD'))
fig.add_trace(go.Scatter(x=steps_newton, y=gap_newton, mode='lines+markers', name='Newton'))
fig.update_layout(
    title='Suboptimality f(x_k) - p* vs. iteration',
    xaxis_title='iteration k',
    yaxis_title='f(x_k) - p* (log scale)',
)
fig.update_yaxes(type='log')
fig.show()

# Effect of the BLS parameters $\alpha$ and $\beta$

We sweep over a grid of backtracking parameters and record how many iterations each method needs to reach the stopping criterion ($\|\nabla f\|_2^2 \le \delta$ for GD, $\lambda^2 \le \delta$ for Newton).

Recall the roles of the parameters:
- $\alpha \in (0, 0.5)$ controls how strict the Armijo *sufficient-decrease* condition is. Larger $\alpha$ demands more decrease per step, so the accepted steps are smaller.
- $\beta \in (0,1)$ is the *shrink factor* during backtracking. Small $\beta$ shrinks aggressively (coarse step granularity); $\beta$ close to 1 shrinks slowly (fine granularity, but more backtracking evaluations).

We run the sweep on a **smaller instance** so the (many) runs finish quickly — the qualitative effect of $\alpha,\beta$ on the iteration count does not depend on the problem size.

In [19]:
# Smaller instance for the parameter sweep (fast to run many times).
m_exp, n_exp = 500, 300
np.random.seed(0)
A_exp = np.random.rand(m_exp, n_exp)
obj_exp = ObjectiveFunction(A_exp)
x0_exp = np.zeros(n_exp)

alphas = [0.01, 0.1, 0.25, 0.4]
betas = [0.2, 0.5, 0.8]

In [20]:
# Gradient descent: iterations to converge for each (alpha, beta)
print("Gradient descent - iterations to converge (tol=1e-12):")
print(f"{'alpha':>8} {'beta':>8} {'iters':>8}")
for alpha in alphas:
    for beta in betas:
        _, crit, _, _ = gd_bls(obj_exp, x0_exp, alpha=alpha, beta=beta)
        print(f"{alpha:>8} {beta:>8} {len(crit):>8}")

Gradient descent - iterations to converge (tol=1e-12):
   alpha     beta    iters
    0.01      0.2     1000
    0.01      0.5       41
    0.01      0.8       81
     0.1      0.2       53
     0.1      0.5       40
     0.1      0.8       63
    0.25      0.2       45
    0.25      0.5       16
    0.25      0.8       24
     0.4      0.2       41
     0.4      0.5       20
     0.4      0.8       17


In [21]:
# Newton's method: iterations to converge for each (alpha, beta)
print("Newton's method - iterations to converge (tol=1e-12):")
print(f"{'alpha':>8} {'beta':>8} {'iters':>8}")
for alpha in alphas:
    for beta in betas:
        _, crit, _, _ = newton_bls(obj_exp, x0_exp, alpha=alpha, beta=beta)
        print(f"{alpha:>8} {beta:>8} {len(crit):>8}")

Newton's method - iterations to converge (tol=1e-12):
   alpha     beta    iters
    0.01      0.2       11
    0.01      0.5       11
    0.01      0.8       11
     0.1      0.2       11
     0.1      0.5       11
     0.1      0.8       11
    0.25      0.2       11
    0.25      0.5       11
    0.25      0.8       11
     0.4      0.2       11
     0.4      0.5       11
     0.4      0.8       11


### Observations

**Gradient descent is sensitive to $\alpha,\beta$.** Across the grid the iteration count ranges from $16$ to "did not converge in $1000$ iterations" (the $\alpha=0.01,\ \beta=0.2$ corner — note that a reported count of `1000` means the run hit `max_iters`, *not* that it converged there). The fewest iterations occur for a moderately strict Armijo condition with a balanced shrink factor (around $\alpha \in [0.25, 0.4]$, $\beta = 0.5$). The trend is not perfectly monotone — it interacts with the conditioning of the problem — which is exactly why $\alpha,\beta$ are parameters one has to tune for GD.

**Newton's method is essentially immune to $\alpha,\beta$.** It takes $11$ iterations for *every* $(\alpha,\beta)$ pair in the grid. The reason: once Newton reaches the quadratically-convergent region it takes full unit steps ($t=1$), which satisfy the Armijo condition for any reasonable $\alpha$, and the short initial damped phase contributes only a couple of iterations. So the line-search parameters have almost no leverage on the total count. This robustness is due Newton's affine invariance.

## `cvxpy` check

In [22]:
x_cp = cp.Variable(n)

first_term_cp = -cp.sum(cp.log(np.ones(m) - A @ x_cp))
second_term_cp = -cp.sum(cp.log(np.ones(n) - cp.square(x_cp)))
objective_cp = cp.Minimize(first_term_cp + second_term_cp)

problem = cp.Problem(objective_cp)

In [23]:
# Uncomment to verify the solution. NOTE: Takes very long!
# problem.solve()
# problem.solution